<a href="https://colab.research.google.com/github/IsuruKRanasundara/AngularProject/blob/main/ChatBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#install Necessary Libraries
!pip install transformers torch

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
chat_history_ids = None
attention_mask_history = None

def chat_with_bot(user_input):
    global chat_history_ids
    global attention_mask_history

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    )

    attention_mask_new_input = torch.ones(new_input_ids.shape, dtype=torch.long)

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        attention_mask_current = torch.cat([attention_mask_history, attention_mask_new_input], dim=-1)
    else:
        bot_input_ids = new_input_ids
        attention_mask_current = attention_mask_new_input

    output = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        attention_mask=attention_mask_current
    )

    chat_history_ids = output
    attention_mask_history = torch.ones(chat_history_ids.shape, dtype=torch.long)

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response

In [6]:
print("Chatbot is ready! Type 'quit' to stop.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == "quit":
        break
    reply = chat_with_bot(user_input)
    print("Bot:", reply)



Chatbot is ready! Type 'quit' to stop.

You: hello
Bot: I think it's a cricket team
You: hello
Bot: I think it's a cricket team


KeyboardInterrupt: Interrupted by user